# Clase de Preprocesamiento del Corpus 
## 06 de mayo 2026

In [ ]:
import os
import math
import re
import string
from collections import Counter, defaultdict

ruta_libros = os.path.join(os.getcwd(), '100libros')

if not os.path.exists(ruta_libros):
    print(f"Error: No se encontró la carpeta en {os.path.abspath(ruta_libros)}")
else:
    archivos = sorted(os.listdir(ruta_libros))
    print(f'Libros disponibles: {len(archivos)}')

    corpus_gutenberg = []
    nombres_gutenberg = []

    for archivo in archivos:
        if archivo.endswith('.txt'):
            ruta_archivo = os.path.join(ruta_libros, archivo)
            try:
                with open(ruta_archivo, 'r', encoding='utf-8') as f:
                    texto = f.read()
                corpus_gutenberg.append(texto)
                nombres_gutenberg.append(archivo)
            except Exception as e:
                print(f'  Error al leer {archivo}: {e}')

    print(f'Libros cargados: {len(corpus_gutenberg)}')

Libros disponibles: 100
Libros cargados: 100


Carga del corpus

In [12]:
import nltk
from nltk.corpus import stopwords

# nltk.download('stopwords')

STOPWORDS_NLTK = set(stopwords.words('spanish'))

print(f'Stopwords en NLTK: {len(STOPWORDS_NLTK)} palabras')

Stopwords en NLTK: 313 palabras


In [15]:
def limpiar_texto(texto):
    texto = texto.lower()
    # Eliminar puntuación y caracteres especiales (dejar solo letras y espacios)
    texto = re.sub(r'[^a-záéíóúüñ\s]', '', texto)
    # Eliminar espacios múltiples
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto


In [22]:
def tokenizar(texto, stopwords=STOPWORDS_NLTK):
    texto_limpio = limpiar_texto(texto)
    tokens = texto_limpio.split()
    tokens = [t for t in tokens if t not in stopwords and len(t) > 2]

    return tokens


Limpieza y tokenizacion del corpus

In [24]:
corpus_tokenizado = [tokenizar(doc) for doc in corpus_gutenberg]
total_tokens = sum(len(doc) for doc in corpus_tokenizado)
print(f"Total de tokens: {total_tokens:,}")

Total de tokens: 5,780,733


Stemming

In [31]:
from nltk.stem import SnowballStemmer
stemmer = SnowballStemmer('spanish')

corpus_stemmed = [
    [stemmer.stem(token) for token in doc] 
    for doc in corpus_tokenizado
]
print(f"Original: {corpus_tokenizado[0][:1000]}")
print(f"Stemmed: {corpus_stemmed[0][:1000]}")

Original: ['the', 'project', 'gutenberg', 'ebook', 'first', 'spanish', 'reader', 'this', 'ebook', 'for', 'the', 'use', 'anyone', 'anywhere', 'the', 'united', 'states', 'and', 'most', 'other', 'parts', 'the', 'world', 'cost', 'and', 'with', 'almost', 'restrictions', 'whatsoever', 'you', 'may', 'copy', 'give', 'away', 'reuse', 'under', 'the', 'terms', 'the', 'project', 'gutenberg', 'license', 'included', 'with', 'this', 'ebook', 'online', 'wwwgutenbergorg', 'you', 'are', 'not', 'located', 'the', 'united', 'states', 'you', 'will', 'have', 'check', 'the', 'laws', 'the', 'country', 'where', 'you', 'are', 'located', 'before', 'using', 'this', 'ebook', 'title', 'first', 'spanish', 'reader', 'author', 'erwin', 'roessler', 'alfred', 'remy', 'release', 'date', 'march', 'ebook', 'most', 'recently', 'updated', 'december', 'language', 'english', 'spanish', 'other', 'information', 'and', 'formats', 'wwwgutenbergorgebooks', 'credits', 'produced', 'john', 'hagerson', 'kevin', 'handy', 'renald', 'leves

Lemmatization como ya hicimos el uno, este ya no hacemos

In [ ]:
#import spacy
#
## Desactivamos componentes innecesarios (parser, ner) para ganar mucha velocidad
#nlp = spacy.load("es_core_news_sm", disable=["parser", "ner"])
#
#def lematizar_corpus(corpus_tokens):
#    corpus_lemas = []
#    
#    # Unimos los tokens de nuevo en strings para que SpaCy los procese
#    textos_unidos = [" ".join(doc) for doc in corpus_tokens]
#    
#    # nlp.pipe es la forma más rápida de procesar grandes volúmenes
#    for doc_spacy in nlp.pipe(textos_unidos, batch_size=10, n_process=-1):
#        # Extraemos el lema de cada palabra
#        lemas = [token.lemma_ for token in doc_spacy]
#        corpus_lemas.append(lemas)
#        
#    return corpus_lemas
#
## Nota: Esto puede tardar unos minutos debido al volumen de 5.7M de tokens
#corpus_lematizado = lematizar_corpus(corpus_tokenizado)

El profesor dijo que ahora usaremos un dataframe, entonces del corpus debo sacarlo a un dataframe que tenga el numero de documento, el raw del texto y el processed.
Donde el processed se llenara con una funcion que haga todo, limpiar tokenizar y el stemming, y devolvera a la columna processed del dataframe.

Primero el dataframe con pandas

In [36]:
import pandas as pd

df = pd.DataFrame({
    'raw': corpus_gutenberg
})
df.insert(0, 'id_doc', range(1, len(df) + 1))
print(df.head())

   id_doc                                                raw
0       1  ﻿The Project Gutenberg eBook of A First Spanis...
1       2  ﻿The Project Gutenberg eBook of Adán y Eva en ...
2       3  ﻿The Project Gutenberg eBook of Amar es vencer...
3       4  ﻿The Project Gutenberg eBook of Amaury\n\n    ...
4       5  ﻿The Project Gutenberg eBook of An Elementary ...


In [37]:
def procesar(texto):
    tokens = tokenizar(texto) 
    stems = [stemmer.stem(t) for t in tokens]
    # 3. Devolver como un string unido por espacios 
    return " ".join(stems)

In [39]:
df['processed'] = df['raw'].apply(procesar)

print(df.head())

   id_doc                                                raw  \
0       1  ﻿The Project Gutenberg eBook of A First Spanis...   
1       2  ﻿The Project Gutenberg eBook of Adán y Eva en ...   
2       3  ﻿The Project Gutenberg eBook of Amar es vencer...   
3       4  ﻿The Project Gutenberg eBook of Amaury\n\n    ...   
4       5  ﻿The Project Gutenberg eBook of An Elementary ...   

                                           processed  
0  the project gutenberg ebook first spanish read...  
1  the project gutenberg ebook adan eva parais th...  
2  the project gutenberg ebook amar venc this ebo...  
3  the project gutenberg ebook amaury this ebook ...  
4  the project gutenberg ebook elementary spanish...  


Ahora agregaremos una columna que da el tfidfd de cada fila, osea el vector

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizador = TfidfVectorizer(lowercase=False, stop_words=list(STOPWORDS_NLTK))

tfidf_matrix = vectorizador.fit_transform(df['processed'])

df['tfidf_vector'] = tfidf_matrix.toarray().tolist()

print(df.head())

   id_doc                                                raw  \
0       1  ﻿The Project Gutenberg eBook of A First Spanis...   
1       2  ﻿The Project Gutenberg eBook of Adán y Eva en ...   
2       3  ﻿The Project Gutenberg eBook of Amar es vencer...   
3       4  ﻿The Project Gutenberg eBook of Amaury\n\n    ...   
4       5  ﻿The Project Gutenberg eBook of An Elementary ...   

                                           processed  \
0  the project gutenberg ebook first spanish read...   
1  the project gutenberg ebook adan eva parais th...   
2  the project gutenberg ebook amar venc this ebo...   
3  the project gutenberg ebook amaury this ebook ...   
4  the project gutenberg ebook elementary spanish...   

                                        tfidf_vector  
0  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  
1  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  
2  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  
3  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  
4  

Y ahora haremos una query que diga 

query_tfidf = vectorizador.transform(query)

In [42]:
query_usuario = "Don Quijote"

In [44]:
query_procesada = procesar(query_usuario)
print(f"Query original: {query_usuario}")
print(f"Query procesada: {query_procesada}")

Query original: Don Quijote
Query procesada: don quijot


In [51]:
# NOTA: Pasamos la query dentro de una lista [] porque transform espera una colección
query_tfidf = vectorizador.transform([query_procesada])
print(f"Vector de la búsqueda generado. Tamaño: {query_tfidf.shape}")

Vector de la búsqueda generado. Tamaño: (1, 178696)


Hacemos la busqueda

Mediremos la distancia entre la query vector, y cada uno de esos vectores que estan almacenados en el dataframe, esto usando 

No sumarle al dataframe, sino usar la estrcutura de datos array. Lo conecto con los indices 



In [52]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Convertimos la columna de vectores en una matriz que la función entienda
vectores_libros = np.array(df['tfidf_vector'].tolist())

dist = cosine_similarity(vectores_libros, query_tfidf)

df['similitud'] = dist.flatten()

In [53]:
df_final = df.sort_values(by='similitud', ascending=False)

In [56]:
print("Los libros más parecidos a tu búsqueda son:")
print(df[['id_doc', 'similitud']].head(5))

Los libros más parecidos a tu búsqueda son:
   id_doc  similitud
0       1   0.000000
1       2   0.050903
2       3   0.003895
3       4   0.001193
4       5   0.005864


TF Y DF

En sklearn se crea solito el vocabulario limpiado y tokenizado, pero no hace steaming, por eso hicimos antes todo lo anterior

In [ ]:
# 1. Unimos tus tokens ya procesados y stemmed (los que ya no tienen basura ni números)
corpus_listo_para_tfidf = [" ".join(doc) for doc in corpus_stemmed]

# 2. El vectorizador ahora solo calcula pesos, ya no tiene que limpiar casi nada
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizador = TfidfVectorizer(lowercase=False, stop_words=list(STOPWORDS_NLTK))
tfidf_matrix = vectorizador.fit_transform(corpus_listo_para_tfidf)